In [2]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import os, warnings
from datetime import datetime, timedelta


In [3]:
df = pd.read_csv("../data/raw/rewards_synthetic.csv")

In [6]:
# Tampilkan 5 baris pertama dari gabungan data
display(df.head())
print(f"\nShape: {df.shape[0]:,} rows × {df.shape[1]} cols")
print(f"Kolom: {df.columns.tolist()}")

,user_id,segment_true,last_tx_date,recency_days,frequency_monthly,monetary_monthly,ovo_points_balance,tier,services_used,n_services,city,churn_label
0,USR006252,Promising,2024-12-16,16,5,169000.0,1620,Member,GrabExpress,1,Jakarta,0
1,USR004684,At_Risk,2024-10-18,75,4,186000.0,2099,Member,GrabCar|GrabExpress|GrabFood|GrabBike,4,Jakarta,0
2,USR001731,Loyal,2024-12-19,13,9,784000.0,8384,Gold,GrabExpress|GrabCar|GrabFood,3,Jakarta,0
3,USR004742,At_Risk,2024-10-08,85,4,318000.0,3439,Silver,GrabBike,1,Surabaya,0
4,USR004521,At_Risk,2024-11-02,60,4,178000.0,1690,Member,GrabFood|GrabMart|GrabExpress|GrabBike,4,Surabaya,0



Shape: 10,000 rows × 12 cols
Kolom: ['user_id', 'segment_true', 'last_tx_date', 'recency_days', 'frequency_monthly', 'monetary_monthly', 'ovo_points_balance', 'tier', 'services_used', 'n_services', 'city', 'churn_label']


In [4]:
print("=" * 50)
print("INFO DATASET")
print("=" * 50)
print(df.info())
print()
print(df.describe(include="all").T[["count","mean","std","min","max"]].round(2))

INFO DATASET
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 12 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   user_id             10000 non-null  object 
 1   segment_true        10000 non-null  object 
 2   last_tx_date        10000 non-null  object 
 3   recency_days        10000 non-null  int64  
 4   frequency_monthly   10000 non-null  int64  
 5   monetary_monthly    10000 non-null  float64
 6   ovo_points_balance  10000 non-null  int64  
 7   tier                10000 non-null  object 
 8   services_used       10000 non-null  object 
 9   n_services          10000 non-null  int64  
 10  city                10000 non-null  object 
 11  churn_label         10000 non-null  int64  
dtypes: float64(1), int64(5), object(6)
memory usage: 937.6+ KB
None

                      count       mean            std  min        max
user_id               10000        NaN            NaN

In [7]:
# Cek jumlah total baris yang duplikat (semua kolom identik)
total_duplikat = df.duplicated().sum()
print(f"Total data duplikat: {total_duplikat} baris")

# Tampilkan detail jika ada duplikat
if total_duplikat > 0:
    print("\nDetail Data Duplikat:")
    display(df[df.duplicated(keep=False)])
else:
    print("✅ Tidak ada duplikat — data bersih di level baris.")

Total data duplikat: 0 baris
✅ Tidak ada duplikat — data bersih di level baris.


In [10]:
missing     = df.isnull().sum().rename("missing")
missing_pct = (df.isnull().mean() * 100).round(2).rename("pct_missing")
miss_df     = pd.concat([missing, missing_pct], axis=1).sort_values("pct_missing", ascending=False)
print("Ringkasan Missing Value:")
print(miss_df.to_string())

miss_df_filtered = miss_df[miss_df["pct_missing"] > 0]

if not miss_df_filtered.empty:
    fig, ax = plt.subplots(figsize=(9, 4))
    
    # Plot hanya kolom yang ada missing value-nya
    miss_df_filtered["pct_missing"].plot(kind="barh", ax=ax, color="#E74C3C")
    
    ax.set_xlabel("% Missing")
    ax.set_title("Missing Value Rate per Column", fontweight="bold")
    
    # Kunci sumbu X agar dimulai dari 0 dan sedikit lebih panjang dari nilai maksimal
    ax.set_xlim(0, miss_df_filtered["pct_missing"].max() + 5)
    
    # Tambahkan teks persentase
    for i, val in enumerate(miss_df_filtered["pct_missing"]):
        ax.text(val + 0.3, i, f"{val:.1f}%", va="center", fontsize=9)
        
    plt.tight_layout()
    plt.show()
else:
    print("\n🎉 YAY! Dataset kamu 100% BERSIH dari missing value!")

Ringkasan Missing Value:
                    missing  pct_missing
user_id                   0          0.0
segment_true              0          0.0
last_tx_date              0          0.0
recency_days              0          0.0
frequency_monthly         0          0.0
monetary_monthly          0          0.0
ovo_points_balance        0          0.0
tier                      0          0.0
services_used             0          0.0
n_services                0          0.0
city                      0          0.0
churn_label               0          0.0

🎉 YAY! Grafik tidak perlu ditampilkan karena dataset kamu 100% BERSIH dari missing value!


In [17]:
df.describe(include="all")

,user_id,segment_true,last_tx_date,recency_days,frequency_monthly,monetary_monthly,ovo_points_balance,tier,services_used,n_services,city,churn_label
count,10000,10000,10000,10000.000000,10000.000000,1.000000e+04,10000.000000,10000,10000,10000.000000,10000,10000.000000
unique,10000,5,348,NaN,NaN,NaN,NaN,4,205,NaN,5,NaN
top,USR006252,Loyal,2024-12-20,NaN,NaN,NaN,NaN,Member,GrabExpress,NaN,Jakarta,NaN
freq,1,2500,365,NaN,NaN,NaN,NaN,4156,731,NaN,4525,NaN
mean,NaN,NaN,NaN,66.654100,7.752300,4.712105e+05,4706.813300,NaN,NaN,2.102200,NaN,0.343700
std,NaN,NaN,NaN,90.364359,7.436873,5.631958e+05,5683.905669,NaN,NaN,1.007302,NaN,0.474966
min,NaN,NaN,NaN,1.000000,0.000000,0.000000e+00,0.000000,NaN,NaN,1.000000,NaN,0.000000
25%,NaN,NaN,NaN,11.000000,2.000000,1.130000e+05,1105.750000,NaN,NaN,1.000000,NaN,0.000000
50%,NaN,NaN,NaN,21.000000,5.000000,2.410000e+05,2457.500000,NaN,NaN,2.000000,NaN,0.000000
75%,NaN,NaN,NaN,78.000000,12.000000,5.980000e+05,5912.000000,NaN,NaN,3.000000,NaN,1.000000
